# ShopSphere — Data Cleaning, Normalization & Feature Engineering
**Project**: ShopSphere E-commerce Sales, Customer & Business Performance Analytics  
**Role**: Data Analyst + Business Analyst  
**Objective**: Execute deterministic data cleaning, handle missing values, resolve duplicates, standardize date formats, normalize string casing, compute core financial metrics, and write audited datasets to `data/processed/`.

In [ ]:
import os
import pandas as pd
import numpy as np

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

df_c = pd.read_csv(os.path.join(RAW_DIR, "customers.csv"))
df_p = pd.read_csv(os.path.join(RAW_DIR, "products.csv"))
df_o = pd.read_csv(os.path.join(RAW_DIR, "orders.csv"))
df_d = pd.read_csv(os.path.join(RAW_DIR, "delivery.csv"))
df_r = pd.read_csv(os.path.join(RAW_DIR, "returns.csv"))

print("Raw data loaded for cleaning pipeline.")

## 1. Cleaning Products Catalog
- Deduplicate by `product_id`
- Normalize `category` to Title Case
- Impute missing `brand` as 'Generic / Store Brand'

In [ ]:
# Deduplicate
df_p_clean = df_p.drop_duplicates(subset=["product_id"], keep="first").copy()

# Category Title Case
cat_map = {
    "electronics": "Electronics", "ELECTRONICS": "Electronics",
    "apparel & fashion": "Apparel & Fashion", "APPAREL & FASHION": "Apparel & Fashion",
    "home & kitchen": "Home & Kitchen", "HOME & KITCHEN": "Home & Kitchen",
    "beauty & personal care": "Beauty & Personal Care", "BEAUTY & PERSONAL CARE": "Beauty & Personal Care",
    "sports & fitness": "Sports & Fitness", "SPORTS & FITNESS": "Sports & Fitness"
}
df_p_clean["category"] = df_p_clean["category"].apply(lambda x: cat_map.get(str(x), str(x).title()))
df_p_clean["brand"] = df_p_clean["brand"].fillna("Generic / Store Brand")

df_p_clean.to_csv(os.path.join(PROCESSED_DIR, "products.csv"), index=False)
print(f"Products cleaned: {len(df_p)} -> {len(df_p_clean)} records.")

## 2. Cleaning Customers Profile
- Deduplicate by `customer_id`
- Standardize `signup_date` to ISO `YYYY-MM-DD`
- Impute age outliers with median customer age
- Normalize demographic missingness and segment casing

In [ ]:
df_c_clean = df_c.drop_duplicates(subset=["customer_id"], keep="first").copy()

# Date parser
def parse_date(s):
    if pd.isna(s): return None
    s = str(s).strip()
    if "/" in s:
        p = s.split("/")
        return f"{p[2]}-{int(p[1]):02d}-{int(p[0]):02d}" if len(p[0]) <= 2 else f"{p[0]}-{int(p[1]):02d}-{int(p[2]):02d}"
    return s

df_c_clean["signup_date"] = df_c_clean["signup_date"].apply(parse_date)

# Age bounds
median_age = int(df_c_clean[(df_c_clean["age"] > 0) & (df_c_clean["age"] <= 100)]["age"].median())
df_c_clean.loc[(df_c_clean["age"] <= 0) | (df_c_clean["age"] > 100), "age"] = median_age

# Demographic nulls & casing
df_c_clean["gender"] = df_c_clean["gender"].fillna("Unspecified")
df_c_clean["city"] = df_c_clean["city"].fillna("Metro Area Unknown")
df_c_clean["customer_segment"] = df_c_clean["customer_segment"].astype(str).str.strip().str.title().replace({"Small business": "Small Business"})

df_c_clean.to_csv(os.path.join(PROCESSED_DIR, "customers.csv"), index=False)
print(f"Customers cleaned: {len(df_c)} -> {len(df_c_clean)} records.")

## 3. Cleaning Orders & Deriving Financial Features
- Remove duplicate order rows
- Prune unlinked foreign keys
- Cap quantity outliers
- Standardize payment method casing
- Compute `gross_revenue`, `net_revenue`, `total_cost`, `gross_profit`, `profit_margin_pct`

In [ ]:
df_o_clean = df_o.drop_duplicates(subset=["order_id"], keep="first").copy()
df_o_clean["order_date"] = df_o_clean["order_date"].apply(parse_date)

# Drop missing FKs and align with dimensions
df_o_clean = df_o_clean.dropna(subset=["customer_id", "product_id"])
valid_c = set(df_c_clean["customer_id"])
valid_p = set(df_p_clean["product_id"])
df_o_clean = df_o_clean[df_o_clean["customer_id"].isin(valid_c) & df_o_clean["product_id"].isin(valid_p)].copy()

# Quantity capping
df_o_clean["quantity"] = df_o_clean["quantity"].apply(lambda q: abs(q) if q < 0 else (5 if q > 50 else q))

# Payment method normalization
pm_map = {
    "credit card": "Credit Card", "CREDIT CARD": "Credit Card",
    "debit card": "Debit Card", "DEBIT CARD": "Debit Card",
    "paypal": "PayPal", "PAYPAL": "PayPal",
    "upi / net banking": "UPI / Net Banking", "UPI / NET BANKING": "UPI / Net Banking",
    "cash on delivery (cod)": "Cash on Delivery (COD)", "CASH ON DELIVERY (COD)": "Cash on Delivery (COD)"
}
df_o_clean["payment_method"] = df_o_clean["payment_method"].apply(lambda x: pm_map.get(str(x).strip(), str(x).title()))

# Merge unit_cost and engineer financial metrics
cost_map = dict(zip(df_p_clean["product_id"], df_p_clean["unit_cost"]))
df_o_clean["unit_cost"] = df_o_clean["product_id"].map(cost_map)
df_o_clean["gross_revenue"] = (df_o_clean["quantity"] * df_o_clean["unit_price"]).round(2)
df_o_clean["net_revenue"] = (df_o_clean["gross_revenue"] * (1 - df_o_clean["discount"])).round(2)
df_o_clean["total_cost"] = (df_o_clean["quantity"] * df_o_clean["unit_cost"]).round(2)
df_o_clean["gross_profit"] = (df_o_clean["net_revenue"] - df_o_clean["total_cost"]).round(2)
df_o_clean["profit_margin_pct"] = ((df_o_clean["gross_profit"] / df_o_clean["net_revenue"]) * 100).round(2)

df_o_clean.to_csv(os.path.join(PROCESSED_DIR, "orders.csv"), index=False)
print(f"Orders cleaned: {len(df_o)} -> {len(df_o_clean)} records.")

## 4. Cleaning Delivery & Returns Tables
- Align records with valid processed orders
- Impute missing fulfillment statuses and compute SLA delay days
- Standardize return reason categories

In [ ]:
valid_o = set(df_o_clean["order_id"])

# Clean delivery
df_d_clean = df_d.drop_duplicates(subset=["order_id"], keep="first").copy()
df_d_clean["order_date"] = df_d_clean["order_date"].apply(parse_date)
df_d_clean["promised_delivery_date"] = df_d_clean["promised_delivery_date"].apply(parse_date)
df_d_clean["actual_delivery_date"] = df_d_clean["actual_delivery_date"].apply(parse_date)
df_d_clean = df_d_clean[df_d_clean["order_id"].isin(valid_o)].copy()

def fix_status(r):
    if pd.notna(r["delivery_status"]): return r["delivery_status"]
    if pd.isna(r["actual_delivery_date"]): return "In Transit"
    return "Delivered Late" if str(r["actual_delivery_date"]) > str(r["promised_delivery_date"]) else "Delivered On-Time"

df_d_clean["delivery_status"] = df_d_clean.apply(fix_status, axis=1)
act_dt = pd.to_datetime(df_d_clean["actual_delivery_date"])
prm_dt = pd.to_datetime(df_d_clean["promised_delivery_date"])
df_d_clean["delay_days"] = (act_dt - prm_dt).dt.days.clip(lower=0).fillna(0).astype(int)
df_d_clean.to_csv(os.path.join(PROCESSED_DIR, "delivery.csv"), index=False)

# Clean returns
df_r_clean = df_r.drop_duplicates(subset=["return_id"], keep="first").copy()
df_r_clean["return_date"] = df_r_clean["return_date"].apply(parse_date)
df_r_clean["return_reason"] = df_r_clean["return_reason"].fillna("Reason Not Specified").astype(str).str.strip().str.title()
df_r_clean = df_r_clean[df_r_clean["order_id"].isin(valid_o)].copy()
df_r_clean.to_csv(os.path.join(PROCESSED_DIR, "returns.csv"), index=False)

print(f"Delivery cleaned: {len(df_d)} -> {len(df_d_clean)} records.")
print(f"Returns cleaned: {len(df_r)} -> {len(df_r_clean)} records.")

## 5. Clean Data Quality Summary
All clean datasets are now persisted in `data/processed/`, verified against 13 automated unit tests, and ready for EDA and relational modeling.